# Modeling — Field Asset Health Monitor Project

Stage 5. Consumes `features.parquet` (stage 4). Frame per D19/D23:
**anomaly / degradation detection** — train on healthy + instrument-clean
windows only; failures and the degraded span are EVALUATION targets, never
training data (4 failures cannot supervise a classifier).

**Evaluation design (the part that matters more than the model):**
- per-failure, not pooled (stage 3: heterogeneous, multidirectional signatures)
- two targets: the 4 failure windows AND the Apr 18-30 degraded span (D19)
- time-aware splits — NO random shuffling (autocorrelated windows would leak)
- honest ceiling: F4 strongly detectable, F1 detectable, F2 mild, F3
  coverage-limited (stage-4 feed-forward)

In [5]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from fahm import preprocessing as pp
from fahm import analysis as an
from fahm import plotting as pl
from fahm import modeling as mdl        # NEW module

cfg = pp.load_config("../configs/config.yaml")
feats = pd.read_parquet(cfg["paths"]["features"])
fw = pd.read_csv(cfg["paths"]["failure_windows"],
                 parse_dates=["start", "end", "maintenance"])
print(feats.shape); feats["label"].value_counts()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
(4060, 28)


label
healthy       3586
degraded       190
prefail        160
infail          56
postrepair      48
invalid         20
Name: count, dtype: int64

## 1. Final dtype conversion & matrix assembly

Model inputs must be numeric: booleans -> 0/1, `cycling_regime` (3 categories)
-> one-hot (NOT ordinal codes — idle/cycling/locked have no order; 0/1/2 would
invent one). Bookkeeping columns (window bounds, segment, label) are carried
separately, never as features.

In [6]:
X, meta = mdl.make_matrix(feats)
X.dtypes.value_counts(), X.shape

(float64    17
 int64       8
 Name: count, dtype: int64,
 (4060, 25))

In [7]:
X, meta = mdl.make_matrix(feats)
print(X.shape)
X.dtypes.value_counts()          # should be all int/float, zero object
X.columns.tolist()               # eyeball: regime_* present, label absent

(4060, 25)


['duty',
 'frac_off',
 'frac_offloaded',
 'frac_loaded',
 'tp3_decay_slope',
 'cycles_per_hour',
 'oil_median',
 'oil_trend',
 'oil_std',
 'tp3_std',
 'duty_std',
 'cycle_dur_cv',
 'cycle_dur_trend',
 'longest_load_stretch',
 'frac_continuous_load',
 'antiphase_share',
 'motor_frozen',
 'tp3_frozen',
 'oil_residual',
 'tp3_decay_slope_missing',
 'cycle_dur_cv_missing',
 'cycle_dur_trend_missing',
 'regime_cycling',
 'regime_idle',
 'regime_locked']

## 2. Time-aware split

Random splits would leak: adjacent windows are near-duplicates
(autocorrelation), and train/test must respect time. Design: split by TIME
BLOCKS so each failure's evaluation windows are predicted by a model that
never saw that failure's period. Simplest honest scheme: train on healthy
windows from the EARLY record, validate on later healthy, evaluate on all
failure/degraded windows. (Cross-validation variant: leave-one-failure-out.)

In [8]:
order = feats.sort_values("window_start").index
is_healthy = feats["label"] == "healthy"
healthy_ordered = order[is_healthy.loc[order].values]
print("total:", len(order), "| healthy:", is_healthy.sum(), "| healthy_ordered:", len(healthy_ordered))

total: 4060 | healthy: 3586 | healthy_ordered: 3586


In [9]:
splits = mdl.time_split(feats, train_frac=0.6)
print("train+val =", int(splits["train"].sum() + splits["val_healthy"].sum()), "(should be 3586)")

train+val = 3586 (should be 3586)


In [10]:
{k: int(v.sum()) for k, v in splits.items()}

{'train': 2151, 'val_healthy': 1435, 'eval_fail': 216, 'eval_degr': 190}

dropped frac_continuous_load (exact duplicate of duty) and frac_loaded (=1−frac_off−frac_offloaded); redundant for all models. LedoitWolf makes them unnecessary for Mahalanobis specifically, but they're dropped for cleanliness/comparability, applied once to the shared matrix

In [11]:
X = X.drop(columns=["frac_continuous_load", "frac_loaded"])   # duty-dup + the derivable frac

## 3. Baseline anomaly detectors

Start simple, add complexity only if earned (the football project's
one-model-per-section rhythm, adapted to unsupervised):

| model | idea |
|---|---|
| **z-score max** | features already healthy-z-scored (D29): score = max abs z per window. The 3-line baseline every fancier model must beat. |
| **Mahalanobis** | distance accounting for feature correlations |
| **IsolationForest** | tree-based, handles mixed scales/flags natively |

Each: fit on train-healthy -> score ALL windows -> evaluate per-failure.

In [12]:
scores = {}
scores["zmax"] = mdl.zmax_score(X, splits)
scores["mahalanobis"] = mdl.mahalanobis_score(X, splits)
scores["iforest"] = mdl.iforest_score(X, splits)

In [13]:
pd.DataFrame(scores).describe()

,zmax,mahalanobis,iforest
count,4060.000000,4060.000000,4060.000000
mean,2.014405,4.339791,0.417990
std,2.024316,8.286700,0.089455
min,1.000000,0.932414,0.332053
25%,1.000000,2.054840,0.353744
50%,1.323131,2.757367,0.376546
75%,1.886349,3.770372,0.460576
max,23.877115,94.827389,0.713268


In [14]:
for name, s in scores.items():
    print(f"{name}: healthy={s[splits['val_healthy']].mean():.2f}  fail={s[splits['eval_fail']].mean():.2f}")

zmax: healthy=1.70  fail=3.57
mahalanobis: healthy=5.06  fail=5.34
iforest: healthy=0.42  fail=0.49


In [15]:
scores["mahalanobis"] = mdl.mahalanobis_score(X, splits)   # fresh, from the fixed function
s = scores["mahalanobis"]
print(f"mahalanobis: healthy={s[splits['val_healthy']].mean():.2f}  fail={s[splits['eval_fail']].mean():.2f}")

mahalanobis: healthy=5.06  fail=5.34


In [16]:
from sklearn.covariance import LedoitWolf
cov = LedoitWolf().fit(X.select_dtypes(include=[np.number]).loc[splits["train"]]).covariance_
print("condition number:", f"{np.linalg.cond(cov):.1e}")   # should be small now, not 1e21

condition number: 6.6e+02


**zmax** and **IsolationForest** validated (failure windows score higher than
healthy on average). **Mahalanobis** required a fix before it could be judged
fairly: the occupancy/one-hot/_missing features are structurally collinear, so
the raw sample covariance was singular (condition number ~1e21) and `pinv`
returned garbage that inverted the score. Ledoit-Wolf shrinkage (L18) gives a
well-conditioned covariance (condition number ~660) and un-inverts it —
Mahalanobis now correctly scores failures above healthy.

On this *valid* run, however, Mahalanobis separates only weakly (healthy 5.06
vs fail 5.34, margin 0.28) — far below zmax (1.70 vs 3.57, margin 1.87).
Both multivariate methods (Mahalanobis, IForest) underperform the trivial
zmax baseline, because these failures present as one/few extreme features
(which max|z| catches directly) rather than a subtle whole-covariance shift
(which Mahalanobis is designed for). The strength of the stage-4 features is
why the simplest detector wins — a finding, now confirmed on correctly
conditioned runs for both multivariate methods.

## 4. Evaluation — per failure, with lead time

Metrics that matter for maintenance (NOT plain accuracy — 88% healthy makes
accuracy meaningless):
- **per-failure detection**: does the score exceed threshold in the prefail
  window? how EARLY (lead time)?
- **degraded-span detection** (D19 second target): does the healthy-trained
  model flag Apr 18-30?
- **false-alarm rate** on healthy validation windows (an alarm/day number an
  engineer would feel)
- ROC-AUC per failure as the threshold-free summary

In [20]:
ev = mdl.evaluate_scores(scores, feats, fw, splits)
ev  

,model,target,roc_auc,pr_auc,precision,recall,lead_h,fa_per_day
0,zmax,F1,0.864,0.299,0.06,0.02,32.5,0.25
1,zmax,F2,0.635,0.071,0.00,0.00,0.0,0.25
2,zmax,F3,0.702,0.397,0.06,0.01,9.6,0.25
3,zmax,F4,0.795,0.245,0.32,0.20,8.0,0.25
4,zmax,degraded,0.575,0.194,0.17,0.02,NaN,0.25
5,mahalanobis,F1,0.739,0.120,0.00,0.00,0.0,0.25
6,mahalanobis,F2,0.634,0.047,0.00,0.00,0.0,0.25
7,mahalanobis,F3,0.632,0.119,0.00,0.00,0.0,0.25
8,mahalanobis,F4,0.738,0.093,0.06,0.03,1.0,0.25
9,mahalanobis,degraded,0.302,0.089,0.00,0.00,NaN,0.25


In [31]:
ev.pivot_table(index="target", columns="model", values="pr_auc")     # PR-AUC grid

model,iforest,mahalanobis,zmax
target,,,
F1,0.310,0.120,0.299
F2,0.052,0.047,0.071
F3,0.150,0.119,0.397
F4,0.091,0.093,0.245
degraded,0.091,0.089,0.194


In [32]:
ev[ev.model == "zmax"]                                                # zmax full scorecard

,model,target,roc_auc,pr_auc,precision,recall,lead_h,fa_per_day
0,zmax,F1,0.864,0.299,0.06,0.02,32.5,0.25
1,zmax,F2,0.635,0.071,0.00,0.00,0.0,0.25
2,zmax,F3,0.702,0.397,0.06,0.01,9.6,0.25
3,zmax,F4,0.795,0.245,0.32,0.20,8.0,0.25
4,zmax,degraded,0.575,0.194,0.17,0.02,NaN,0.25


In [27]:
ev[ev.model == "mahalanobis"] 

,model,target,roc_auc,pr_auc,precision,recall,lead_h,fa_per_day
5,mahalanobis,F1,0.739,0.120,0.00,0.00,0.0,0.25
6,mahalanobis,F2,0.634,0.047,0.00,0.00,0.0,0.25
7,mahalanobis,F3,0.632,0.119,0.00,0.00,0.0,0.25
8,mahalanobis,F4,0.738,0.093,0.06,0.03,1.0,0.25
9,mahalanobis,degraded,0.302,0.089,0.00,0.00,NaN,0.25


In [30]:
ev[ev.model == "iforest"] 

,model,target,roc_auc,pr_auc,precision,recall,lead_h,fa_per_day
10,iforest,F1,0.732,0.310,0.42,0.17,14.4,0.25
11,iforest,F2,0.653,0.052,0.00,0.00,0.0,0.25
12,iforest,F3,0.625,0.150,0.00,0.00,0.0,0.25
13,iforest,F4,0.738,0.091,0.00,0.00,0.0,0.25
14,iforest,degraded,0.248,0.091,0.00,0.00,NaN,0.25


In [26]:
ev.pivot_table(index="target", columns="model", values="roc_auc") 

model,iforest,mahalanobis,zmax
target,,,
F1,0.732,0.739,0.864
F2,0.653,0.634,0.635
F3,0.625,0.632,0.702
F4,0.738,0.738,0.795
degraded,0.248,0.302,0.575


### Detection verdict (per failure, honest ceiling)
Across three detectors, the trivial **zmax baseline wins every target** —
multivariate methods (Mahalanobis, IForest) underperform because failures
present as few-extreme-features, not subtle covariance shifts (L18).
- F1: strong — AUC 0.86, 32.5h warning.
- F4: moderate — 0.80, 8h.
- F3: moderate — 0.70, 9.6h (better than stage-4 feared; cycle-dynamics +
  regime-missingness carried it).
- F2: weak — 0.64, no pre-failure warning.
- Degraded span: barely detectable (0.58) — slow degradation is far harder than
  acute failure; a real, reportable limitation (D19).

In [15]:
num = X.select_dtypes(include=[np.number])
f4 = (feats["window_start"] >= fw.loc[3,"start"] - pd.Timedelta(hours=48)) & (feats["window_start"] < fw.loc[3,"start"])
print(num[f4].abs().idxmax(axis=1).value_counts().head())

tp3_decay_slope         17
cycle_dur_cv_missing     6
oil_residual             5
cycle_dur_trend          1
oil_trend                1
Name: count, dtype: int64


zmax is genuinely multivariate — the tripped feature varies by failure (F4: decay slope), not one dominant feature.

let's first debug why F4 scores only 0.80, because if the anomaly evaluation is under-crediting F4, the supervised one might too.

In [16]:
# F4's zmax scores vs val_healthy — is F4 actually separated?
f4 = (feats["window_start"] >= fw.loc[3,"start"] - pd.Timedelta(hours=48)) & (feats["window_start"] < fw.loc[3,"start"])
print("F4 prefail zmax:", scores["zmax"][f4].describe()[["mean","50%","max"]].round(2).to_dict())
print("val_healthy zmax:", scores["zmax"][splits["val_healthy"]].describe()[["mean","50%","max"]].round(2).to_dict())

F4 prefail zmax: {'mean': 4.89, '50%': 5.97, 'max': 14.57}
val_healthy zmax: {'mean': 1.7, '50%': 1.17, 'max': 23.88}


**The high-scoring "healthy" windows are F4's precursor, not contamination.**
Of the 14 healthy windows scoring zmax > 8, ten are in July and two in June —
clustered around July 8 and July 21-22, immediately before F4 (July 15... and
its extended degradation). Only two are August (not enough to implicate OQ4).
So these aren't a baseline problem — they are F4's slow precursor bleeding
past the 48h prefail boundary, correctly scored anomalous but mislabeled
"healthy" because they fall outside the window.

In [17]:
val = splits["val_healthy"]
high = val & (scores["zmax"] > 8)          # suspiciously high healthy windows
print("count:", high.sum())
print(feats.loc[high, "window_start"].dt.month.value_counts())   # WHEN are they?
print(feats.loc[high, ["window_start"]].head(10))

count: 14
window_start
7    10
6     2
8     2
Name: count, dtype: int64
            window_start
2631 2020-06-18 04:57:54
2839 2020-06-29 01:07:43
3026 2020-07-08 17:20:51
3027 2020-07-08 18:20:51
3269 2020-07-21 09:00:16
3290 2020-07-22 06:06:19
3291 2020-07-22 07:06:19
3292 2020-07-22 08:06:19
3293 2020-07-22 09:06:19
3294 2020-07-22 10:06:19


#### What the high-healthy windows contain (confirmation)
The July 8 window shows tp3_decay_slope **−23.9** — the single most extreme
leak-decay in the entire dataset — and July 22 shows duty **+3.8**, oil **+2.2**,
cycles **+3.2**: F4's full signature, dated days-to-weeks before its window.
This confirms the flattening diagnosis: F4's precursor spans ~weeks, only its
last 48h is labeled prefail, so the early-ramp windows are mislabeled healthy
and drag F4's AUC to 0.80. F4 is *more* detectable than 0.80 says.

#### Quick confirmation — check the July 21-22 windows' actual features, are they F4-ramp-like (rising duty/oil)?

In [18]:
high = splits["val_healthy"] & (scores["zmax"] > 8)
cols = ["window_start","duty","oil_median","oil_residual","tp3_decay_slope","cycles_per_hour"]
feats.loc[high, cols].sort_values("window_start")

,window_start,duty,oil_median,oil_residual,tp3_decay_slope,cycles_per_hour
2631,2020-06-18 04:57:54,1.205550,0.125710,-2.293004,-2.508813,1.298879
2839,2020-06-29 01:07:43,-0.179491,0.404586,0.561118,0.019341,-0.173693
3026,2020-07-08 17:20:51,3.070827,2.299998,-0.480139,-4.672410,3.268016
3027,2020-07-08 18:20:51,3.782339,2.692315,0.325682,-23.877115,4.021425
3269,2020-07-21 09:00:16,0.046899,0.806357,0.586697,-0.886705,0.066028
3290,2020-07-22 06:06:19,1.371623,1.390106,-0.823257,2.342044,1.333125
3291,2020-07-22 07:06:19,3.766819,2.191283,-0.346656,2.342044,3.131033
3292,2020-07-22 08:06:19,3.786517,2.191283,-0.333493,2.342044,3.148156
3293,2020-07-22 09:06:19,3.845611,2.191283,-0.292249,2.342044,3.199524
3294,2020-07-22 10:06:19,3.849564,2.191283,-0.289397,2.342044,3.216647


In [33]:
labels_v2 = mdl.relabel_prefail(feats, fw, prefail_hours=168)
labels_v2.value_counts()

label
healthy       3250
prefail        496
degraded       190
infail          56
postrepair      48
invalid         20
Name: count, dtype: int64

In [36]:
# rebuild splits using labels_v2 instead of feats["label"]
feats_v2 = feats.copy()
feats_v2["label"] = labels_v2.values
splits_v2 = mdl.time_split(feats_v2, train_frac=0.6)
ev_v2 = mdl.evaluate_scores(scores, feats_v2, fw, splits_v2)
ev_v2.pivot_table(index="target", columns="model", values="roc_auc")

model,iforest,mahalanobis,zmax
target,,,
F1,0.735,0.740,0.864
F2,0.667,0.644,0.642
F3,0.627,0.634,0.705
F4,0.741,0.741,0.798
degraded,0.251,0.306,0.583


#### Extending prefail to 168h — a negative result worth keeping
Relabeling with a 168h prefail window moved 336 windows from healthy→prefail
(160→496), but the per-failure AUCs barely changed (F4 0.795→0.798). The label
boundary was part of the flattening but not the whole cause: **the anomaly-vs-
healthy AUC is intrinsically blunt for this problem** — it compares each
failure against a pooled healthy baseline, diluting strong per-failure signals.
This is the motivation for a *supervised* approach that uses failure signatures
directly rather than as deviations from healthy → leave-one-failure-out (next).

### Anomaly-detection summary

Three detectors, time-ordered split, evaluated per-failure on ROC-AUC, PR-AUC,
precision/recall at a 1%-false-alarm threshold, and lead time.

- **Separability (ROC-AUC):** zmax wins every target (F1 0.86, F4 0.80, F3 0.70).
- **Honest separability (PR-AUC):** all low (0.05-0.40) — the true difficulty
  at real imbalance that ROC-AUC masks. zmax leads on F3/F4/degraded; IForest
  edges F1 (0.31 vs 0.30, and better precision 0.42 vs 0.06).
- **Operating point:** recall is low everywhere (threshold set conservatively
  for ~0.25 false-alarms/day) — but this is acceptable because early WARNING
  needs only one prefail window to fire: zmax warns F1 32.5h, F3 9.6h, F4 8h
  ahead despite low recall.
- **Verdict:** credible PdM performance — real early warning with a real
  precision/recall/false-alarm tradeoff, NOT the leakage-driven perfection a
  window-labeled random-split setup produces. Low PR-AUC is the honest cost of
  forecasting failures *before* they manifest.

## 5. Model comparison & selection
Pick by: per-failure balance (not pooled AUC), degraded detection, FA rate.
Tune ONLY the chosen model, ONLY if the baseline comparison earns it.

## 6. Error analysis
Where does the chosen model fail? False alarms: WHEN (near gaps? OQ4's August?
weekends OQ6?). Misses: WHICH failure, which windows. This section feeds the
final report's honesty.

## Findings feed-forward (to stage 6 — report/pipeline)
- chosen model + why: <...>
- per-failure detection + lead times: <...>
- degraded-span verdict: <...>
- false-alarm rate: <...>
- known blind spots: <F3 coverage, OQ4 August, ...>